# AgentCore Self-Managed Memory Strategy 데모

이 Notebook에서는 boto3로 Amazon Bedrock AgentCore self-managed memory strategy를 설정하고 사용하는 방법을 살펴봅니다. Self-managed memory strategy를 사용하면 대화 event를 trigger로 메모리를 추출하고 통합하는 사용자 지정 pipeline을 만들 수 있습니다.

## 작동 방식

1. Trigger 구성: 단기 메모리 event를 기반으로 pipeline을 호출할 trigger 조건(message 수, idle timeout, token 수) 정의
2. 알림 수신: Trigger 조건이 충족되면 AgentCore가 SNS topic에 알림 게시
3. Payload 처리: AgentCore가 대화 데이터를 S3 bucket으로 전달
4. Memory record 추출 및 저장: 사용자 지정 pipeline이 payload를 검색하고 메모리 처리

Self-managed memory strategy에 관한 자세한 내용은 [AWS 공식 문서](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/memory-self-managed-strategies.html#use-self-managed-strategy)를 참조하세요.

## 설정 개요

이 데모에서는 다음 작업을 수행합니다.
1. 필요한 AWS infrastructure 생성(S3, SNS, SQS, Lambda, IAM role)
2. Self-managed strategy를 사용하는 AgentCore memory 생성
3. Memory 처리 pipeline을 시연하는 test event 생성
4. 저장된 메모리의 검색 및 사용을 시연하는 Agent 생성
5. 완료 후 resource 정리

## 설정 및 Import

In [ ]:
!pip install -r requirements.txt --quiet

In [ ]:
import time
import uuid
from aws_utils import AWSUtils

# AWS region 구성
region_name = "us-east-1"  # 원하는 region으로 변경
aws_utils = AWSUtils(region_name=region_name)

# Lambda function 코드 읽기
with open("lambda_function.py", "r") as f:
    lambda_code = f.read()

## 1단계: Payload 전달용 S3 Bucket 생성

Trigger 조건이 충족되었을 때 AgentCore가 대화 payload를 전달할 S3 bucket을 생성합니다.

In [ ]:
# 고유한 이름으로 S3 bucket 생성
bucket_name = aws_utils.create_s3_bucket("agentcore-memory-payloads")
print(f"S3 bucket created: {bucket_name}")

## 2단계: Memory Job 알림용 SNS Topic 생성

AgentCore가 memory 처리 pipeline을 trigger할 때 알림을 수신할 SNS topic을 생성합니다.

In [ ]:
# SNS topic 생성
sns_topic_name = f"agentcore-memory-notifications-{int(time.time())}"
sns_topic_arn = aws_utils.create_sns_topic(sns_topic_name)
print(f"SNS topic created: {sns_topic_arn}")

## 3단계: SNS Subscription이 있는 SQS Queue 생성

SNS topic을 구독하는 SQS queue를 생성합니다. 이 queue는 Lambda function을 trigger할 memory job 알림을 수신합니다.

In [ ]:
# SQS queue를 생성하고 SNS topic 구독
queue_name = f"agentcore-memory-queue-{int(time.time())}"
queue_url, queue_arn = aws_utils.create_sqs_queue_with_sns_subscription(queue_name, sns_topic_arn)
print(f"SQS queue created: {queue_url}")

## 4단계: IAM Role 생성

다음 두 가지 IAM role을 생성합니다.
1. AgentCore가 S3 및 SNS에 액세스하기 위한 role
2. Lambda가 S3, SQS 및 AgentCore API에 액세스하기 위한 role

In [ ]:
# AgentCore용 IAM role 생성
agentcore_role_name = f"AgentCoreMemoryExecutionRole-{int(time.time())}"
agentcore_role_arn = aws_utils.create_iam_role_for_agentcore(agentcore_role_name, bucket_name, sns_topic_arn)
print(f"AgentCore IAM role created: {agentcore_role_arn}")

# Lambda용 IAM role 생성
lambda_role_name = f"LambdaMemoryProcessingRole-{int(time.time())}"
lambda_role_arn = aws_utils.create_iam_role_for_lambda(lambda_role_name, bucket_name, queue_arn)
print(f"Lambda IAM role created: {lambda_role_arn}")

## 5단계: Memory 처리용 Lambda Function 생성

SQS message로 trigger되는 Lambda function을 생성합니다. 이 function은 다음 작업을 수행합니다.
1. S3에서 대화 payload download
2. Bedrock model로 메모리 추출
3. 추출한 메모리를 AgentCore에 다시 저장

In [ ]:
# Lambda function 생성
function_name = f"agentcore-memory-processor-{int(time.time())}"
function_arn = aws_utils.create_lambda_function(function_name, lambda_role_arn, lambda_code)
print(f"Lambda function created: {function_arn}")

# Lambda에 SQS trigger 추가
event_source_uuid = aws_utils.add_sqs_trigger_to_lambda(function_name, queue_arn)
print(f"SQS trigger added to Lambda: {event_source_uuid}")

## 6단계: Self-Managed Strategy를 사용하는 AgentCore Memory 생성

앞에서 설정한 infrastructure를 사용하는 self-managed strategy 구성으로 AgentCore memory를 생성합니다.

In [ ]:
import importlib
import aws_utils

importlib.reload(aws_utils)

# # 업데이트된 코드로 AWSUtils의 새 instance 생성
aws_utils = aws_utils.AWSUtils(region_name=region_name)

# Self-managed strategy로 memory 생성
memory_name = f"SelfManageMemory{int(time.time())}"
memory_description = "Demo memory using self-managed strategy"

memory_id = aws_utils.create_memory_with_self_managed_strategy(
    memory_name=memory_name,
    memory_description=memory_description,
    role_arn=agentcore_role_arn,
    sns_topic_arn=sns_topic_arn,
    s3_bucket_name=bucket_name,
    message_trigger_count=3,  # Message 3개 후 trigger
    token_trigger_count=500,  # 약 500개 token 후 trigger
    idle_timeout=300,  # 5분간 유휴 상태가 지속되면 trigger
    historical_window_size=5,  # 이전 message 5개를 맥락에 포함
)

print(f"Memory created: {memory_id}")
# print(f"Strategy ID: {strategy_id}")

In [ ]:
def wait_for_memory_to_get_active(memory_id):
    response = aws_utils.agentcore_client_control.get_memory(memoryId=memory_id)

    while response["memory"]["status"] != "ACTIVE":
        time.sleep(30)
        response = aws_utils.agentcore_client_control.get_memory(memoryId=memory_id)
        print(f"Memory creation status: {response['memory']['status']}")
    return response["memory"]["status"]


wait_for_memory_to_get_active(memory_id=memory_id)

## 7단계: Memory Pipeline을 Trigger할 Test Event 생성

Self-managed memory pipeline을 trigger할 test event를 생성하겠습니다. Message trigger count를 초과하도록 충분한 event를 생성합니다.

In [ ]:
actor_id = "test-user-123"

In [ ]:
# Test event 생성
session_id = aws_utils.create_test_events(
    memory_id=memory_id,
    actor_id=actor_id,
    num_events=6,  # Message_trigger_count 3을 초과함
)

print(f"Created test events with session ID: {session_id}")

In [ ]:
aws_utils.agentcore_client.list_events(memoryId=memory_id, actorId=actor_id, sessionId=session_id)

## 8단계: Memory 처리 대기

이제 memory 처리 pipeline이 실행될 때까지 기다려야 합니다. 다음 과정이 포함됩니다.
1. AgentCore가 trigger 조건 감지(message 수 초과)
2. AgentCore가 SNS에 알림 게시
3. SNS가 SQS에 message 전달
4. SQS가 Lambda function trigger
5. Lambda가 대화를 처리하고 메모리 저장

잠시 기다린 후 메모리가 생성되었는지 확인해 보겠습니다.

In [ ]:
print("Waiting 30 seconds for memory processing to complete...")
time.sleep(30)

## 9단계: Memory Record 확인

Memory를 검색하여 memory pipeline이 memory record를 생성했는지 확인해 보겠습니다.

In [ ]:
session_id

In [ ]:
# Memory record 나열
namespace = f"/interests/actor/{actor_id}/session/{session_id}/"


def list_memory_records(memory_id, namespace):
    try:
        response = aws_utils.agentcore_client.list_memory_records(memoryId=memory_id, namespace=namespace)
        print(f"Found {len(response.get('memoryRecordSummaries'))} memory records")

        # 검색 결과 표시
        for idx, result in enumerate(response.get("memoryRecordSummaries")):
            print(f"Memory: {idx}")
            print(f"Content: {result['content']['text']}")
    except Exception as e:
        print(f"Error searching memory: {e}")


list_memory_records(memory_id, namespace)

위 record에는 통합 로직을 추가하지 않았기 때문에 사용자 관심사가 반복해서 표시됩니다. Self-managed strategy를 사용하면 추출과 수집만 수행할지 직접 정의할 수 있으며, 이는 비즈니스 사용 사례에 따라 달라집니다.

In [ ]:
# Memory record 검색
def retrieve_memory_records(memory_id, query, topK, namespace):
    try:
        response = aws_utils.agentcore_client.retrieve_memory_records(
            memoryId=memory_id,
            searchCriteria={"searchQuery": query, "topK": topK},
            namespace=namespace,
        )
        print(f"Found {len(response.get('memoryRecordSummaries'))} memory records")

        # 검색 결과 표시
        for idx, result in enumerate(response.get("memoryRecordSummaries")):
            print(f"\nMemory Record {idx + 1}:")
            print(f"Content: {result['content']['text']}")
    except Exception as e:
        print(f"Error searching memory: {e}")


retrieve_memory_records(memory_id=memory_id, query="food choices for dinner", topK=5, namespace=namespace)

## 10단계: 다른 Content가 포함된 추가 Test Event 생성

다른 content를 포함한 test event를 추가로 생성하여 또 다른 memory 처리 cycle을 trigger하겠습니다.

In [ ]:
# 사용자 지정 test event 생성
session_id = str(uuid.uuid4())
actor_id = "test-user-456"

# 더 구체적인 정보가 포함된 사용자 지정 event
test_events = [
    {
        "user": "I'm trying to eat healthier and have been exploring Mediterranean cuisine lately.",
        "assistant": "That's wonderful! Mediterranean food is both delicious and nutritious. What Mediterranean dishes have you tried so far?",
    },
    {
        "user": "I love Greek salads with feta cheese and olives, and I've been making homemade hummus.",
        "assistant": "Homemade hummus is fantastic! Do you prefer it with tahini or without? And what's your favorite way to serve it?",
    },
    {
        "user": "I always use tahini and like to serve it with fresh vegetables and pita bread. I'm also vegetarian, so I avoid meat.",
        "assistant": "Being vegetarian opens up so many Mediterranean options! Have you tried making stuffed grape leaves or lentil-based dishes?",
    },
    {
        "user": "Not yet, but I'd love to learn. I'm also allergic to shellfish, so I have to be careful with seafood dishes.",
        "assistant": "Good to know about the shellfish allergy. For vegetarian Mediterranean cooking, you might enjoy making moussaka with eggplant or trying some traditional Greek bean dishes. Would you like some recipe suggestions?",
    },
]

# Event 생성
for idx, event in enumerate(test_events):
    try:
        event_payload = [
            {"conversational": {"content": {"text": event["user"]}, "role": "USER"}},
            {
                "conversational": {
                    "content": {"text": event["assistant"]},
                    "role": "ASSISTANT",
                }
            },
        ]

        aws_utils.agentcore_client.create_event(
            memoryId=memory_id,
            actorId=actor_id,
            sessionId=session_id,
            eventTimestamp=int(time.time()),
            payload=event_payload,
            clientToken=str(uuid.uuid4()),
        )

        print(f"Created event {idx + 1}/{len(test_events)}")
        time.sleep(1)

    except Exception as e:
        print(f"Error creating test event: {e}")

print("\nWaiting 30 seconds for memory processing to complete...")
time.sleep(30)

## 11단계: 새 Memory 검색

이제 hiking과 사용자의 반려견에 관한 새 메모리를 검색해 보겠습니다.

In [ ]:
# 야외 활동 관련 memory record 검색
namespace = f"/interests/actor/{actor_id}/session/{session_id}/"
retrieve_memory_records(memory_id=memory_id, query="dog pets golden retriever", topK=5, namespace=namespace)

## 12단계: Agent 생성

이 섹션에서는 hook을 통해 AgentCore Self-Managed Memory와 통합된 Strands agent로 지능형 요리 도우미를 구축하는 방법을 살펴봅니다. 사용자 음식 선호도, 식이 제한, 외식 기록을 장기 메모리에 보관하여 이전 대화와 개인 취향을 바탕으로 개인화된 음식점 추천을 제공하는 데 중점을 둡니다.



In [ ]:
import logging
from typing import Dict

# Logging 설정
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger("customer-support")

# 필요한 module import
from strands import Agent
from strands.hooks import (
    AfterInvocationEvent,
    HookProvider,
    HookRegistry,
    MessageAddedEvent,
)
from bedrock_agentcore.memory import MemoryClient

# MemoryClient 초기화
client = MemoryClient(region_name=region_name)

## 13단계: Self-Managed Memory를 사용하는 요리 도우미용 Memory Hook Provider 생성

Hook은 에이전트 실행 lifecycle의 특정 시점에 실행되는 특수 function입니다. 사용자 지정 hook provider는 self-managed memory strategy를 활용하여 다음 방식으로 요리 맥락을 자동 관리합니다.

- Self-managed memory record에서 **관련 음식 선호도 검색**
- 새 질의에 식이 제한, 요리 선호도, 외식 기록에 관한 **맥락 정보 주입**
- 나중에 참조할 수 있도록 batch operation으로 **외식 상호 작용 저장**

이를 통해 다음과 같은 원활한 메모리 환경을 구현합니다.
- 각 질의를 처리하기 전에 저장된 음식 선호도를 자동 검색
- 외식 기록을 바탕으로 맥락에 맞는 음식점 추천 제공

Self-managed 접근 방식을 사용하면 음식 선호도를 저장하고 검색하며 외식 추천 경험 개선에 활용하는 방식을 완전히 제어할 수 있습니다.


In [ ]:
# Memory strategy 목록에서 namespace를 가져오는 helper function
def get_namespaces(mem_client: MemoryClient, memory_id: str) -> Dict:
    """메모리 전략의 네임스페이스 매핑을 가져옵니다."""
    strategies = mem_client.get_memory_strategies(memory_id)
    return {i["type"]: i["namespaces"][0] for i in strategies}

In [ ]:
class CulinaryAssistantMemoryHooks(HookProvider):
    """요리 어시스턴트 에이전트용 메모리 훅입니다."""

    def __init__(self, memory_id: str, namespace: str):
        self.memory_id = memory_id
        self.namespace = namespace

    def retrieve_food_preferences(self, event: MessageAddedEvent):
        """식사 질의를 처리하기 전에 사용자의 음식 선호도를 검색합니다."""
        messages = event.agent.messages
        if messages[-1]["role"] == "user" and "toolResult" not in messages[-1]["content"][0]:
            user_query = messages[-1]["content"][0]["text"]

            try:
                # Direct API로 음식 선호도 검색
                response = aws_utils.agentcore_client.retrieve_memory_records(
                    memoryId=self.memory_id,
                    searchCriteria={"searchQuery": user_query, "topK": 5},
                    namespace=self.namespace,
                )

                memory_records = response.get("memoryRecordSummaries", [])

                if memory_records:
                    # 검색한 선호도 형식 지정
                    preferences_context = []
                    for record in memory_records:
                        content = record.get("content", {}).get("text", "").strip()
                        if content:
                            preferences_context.append(content)

                    # 질의에 음식 선호도 주입
                    if preferences_context:
                        context_text = "\n".join(preferences_context)
                        original_text = messages[-1]["content"][0]["text"]
                        messages[-1]["content"][0]["text"] = (
                            f"User Food Preferences:\n{context_text}\n\n{original_text}"
                        )
                        logger.info(f"Retrieved {len(preferences_context)} food preference records")

            except Exception as e:
                logger.error(f"Failed to retrieve food preferences: {e}")

    def save_dining_interaction(self, event: AfterInvocationEvent):
        """에이전트 응답 후 식사 추천 상호 작용을 저장합니다."""
        try:
            messages = event.agent.messages
            if len(messages) >= 2 and messages[-1]["role"] == "assistant":
                # 마지막 user 질의와 agent 응답 가져오기
                user_query = None
                agent_response = None

                for msg in reversed(messages):
                    if msg["role"] == "assistant" and not agent_response:
                        agent_response = msg["content"][0]["text"]
                    elif msg["role"] == "user" and not user_query and "toolResult" not in msg["content"][0]:
                        user_query = msg["content"][0]["text"]
                        break

                if user_query and agent_response:
                    # Direct API로 상호 작용 저장

                    # 여기서 create_memory_record API를 사용
                    # aws_utils.agentcore_client.create_memory_record(...)

                    logger.info("Saved dining interaction to memory")

        except Exception as e:
            logger.error(f"Failed to save dining interaction: {e}")

    def register_hooks(self, registry: HookRegistry) -> None:
        """요리 어시스턴트 메모리 훅을 등록합니다."""
        registry.add_callback(MessageAddedEvent, self.retrieve_food_preferences)
        registry.add_callback(AfterInvocationEvent, self.save_dining_interaction)
        logger.info("Culinary assistant memory hooks registered")

## 14단계: 요리 도우미 Agent 생성

In [ ]:
# 요리 도우미용 memory hook 생성
print(memory_id)
culinary_hooks = CulinaryAssistantMemoryHooks(memory_id, namespace)

# 요리 도우미 agent 생성
culinary_agent = Agent(
    hooks=[culinary_hooks],
    model="global.anthropic.claude-haiku-4-5-20251001-v1:0",
    tools=[],  # 필요에 따라 tool 업데이트
    state={"actor_id": actor_id, "session_id": session_id},
    system_prompt="""You are the Culinary Assistant, a sophisticated restaurant recommendation assistant.

PURPOSE:
- Help users discover restaurants based on their preferences
- Remember user preferences throughout the conversation
- Provide personalized dining recommendations

You have access to a Memory tool that enables you to:
- Store user preferences (dietary restrictions, favorite cuisines, budget preferences, etc.)
- Retrieve previously stored information to personalize recommendations""",
)

print("✅ Culinary assistant agent created with memory capabilities")

#### Agent를 사용할 준비가 되었습니다.

### 요리 도우미 시나리오 테스트

In [ ]:
response1 = culinary_agent("what are the food choices for Dinner?")
print(f"Support Agent: {response1}")

## 15단계: Resource 정리

불필요한 비용이 발생하지 않도록 생성한 모든 resource를 정리하겠습니다.

In [ ]:
# 모든 resource 정리
import importlib
import aws_utils

importlib.reload(aws_utils)

# # 업데이트된 코드로 AWSUtils의 새 instance 생성
aws_utils = aws_utils.AWSUtils(region_name=region_name)

# # Auto-discovery로 resource 정리
aws_utils.cleanup_resources(discover_resources=True)
print("All resources have been cleaned up!")

## 요약

이 Notebook에서는 다음 방법을 살펴봤습니다.

1. Self-managed memory에 필요한 AWS infrastructure 설정
2. Self-managed strategy를 사용하는 AgentCore memory 생성
3. Memory 처리용 trigger 조건 구성
4. Lambda 기반 memory 처리 pipeline 구현
5. 샘플 대화로 memory system 테스트
6. 추출된 메모리 검색
7. Self-managed memory를 테스트할 요리 agent 생성
8. 모든 resource 정리

Self-managed memory strategy를 사용하면 메모리 추출을 완전히 제어하여 특정 사용 사례에 맞는 custom pipeline을 구축할 수 있습니다.